### **1. Prepare the workspace**


In [1]:

from pathlib import Path
import hashlib, importlib.util, os, runpy, sys, time, urllib.request, urllib.error

RUN_STARTED = time.perf_counter()
COURSE_REVISION = "fe0c0204e6076a7ac2139b7336485a097343fb8a"
MANIFEST_SHA256 = "86d15e813caf833b7d86512808176388d4123dc3c01c6525ee849a76e8322c59"
SETUP_SHA256 = "79608f8f9559d9e6eb09d51ee541ee12d7e9262ac57d1ca3a948c8ecb14f0f35"
IN_COLAB = importlib.util.find_spec("google") is not None and importlib.util.find_spec("google.colab") is not None
ROOT = Path("/content/tamweel") if IN_COLAB else Path(os.environ.get("TAMWEEL_PROJECT_DIR", ".")).resolve()
setup_path = ROOT / "scripts/setup_colab.py"
setup_path.parent.mkdir(parents=True, exist_ok=True)
if not setup_path.exists() or hashlib.sha256(setup_path.read_bytes()).hexdigest() != SETUP_SHA256:
    setup_url = f"https://raw.githubusercontent.com/almiyead-rgb/sda-dsc-211-student-template/{COURSE_REVISION}/scripts/setup_colab.py"
    for attempt in range(3):
        try:
            with urllib.request.urlopen(setup_url, timeout=45) as response:
                setup_bytes = response.read(200000)
            if hashlib.sha256(setup_bytes).hexdigest() != SETUP_SHA256:
                raise ValueError("Setup checksum mismatch. Reopen the released notebook.")
            setup_path.write_bytes(setup_bytes)
            break
        except (urllib.error.URLError, TimeoutError):
            if attempt == 2:
                raise RuntimeError("Download unavailable. Retry setup or upload the unchanged project files; see READINESS_GUIDE.")
            time.sleep(attempt + 1)
environment = runpy.run_path(str(setup_path))["prepare"](ROOT, COURSE_REVISION, MANIFEST_SHA256)
FAST_MODE, FULL_MODE, SEED, N_JOBS = True, False, 211, 2

Package                 Version
numpy                   2.1.3
pandas                  2.2.3
scipy                   1.16.3
scikit-learn            1.6.1
matplotlib              3.10.0
xgboost                 3.4.1
lightgbm                4.6.0
shap                    0.52.0
numba                   0.61.2
optuna                  4.5.0
Python 3.13.16 | CPU | seed=211 | n_jobs=2 | FAST_MODE=True
Verified course files. Next: run the data and compatibility checks.


### **2. Inspect the project data**

The training dataset contains 10,000 synthetic applications with 22 model features. The challenge dataset contains 2,500 applications and does not include the target variable.

The target, default_within_90d, represents whether a default event occurs within 90 days of the application. Identifiers, dates, and split-control fields are excluded from the model features.

Expected result: Confirm the dataset sizes, missing values, required columns, data hashes, and challenge-label boundary. The validation stops if the data structure, hashes, or challenge target boundary does not match the expected contract.


In [2]:
import json
import pandas as pd
from IPython.display import display
from data_checks import load_course_data

frames, contract, data_summary = load_course_data(ROOT)
train = frames["train"]
feature_names = [item["name"] for item in contract["features"]]
target_name = contract["target"]["name"]
display(pd.DataFrame(data_summary).T.fillna("—"))
display(train[["application_id", "application_date", "income_sar", "bureau_score", target_name]].head(4))
(ROOT / "artifacts/data_check.json").write_text(json.dumps(data_summary, indent=2), encoding="utf-8")
print("Data checks passed | اجتازت البيانات الفحوص")

,rows,predictors,missing_feature_values,positive_rate
train,10000.0,22.0,766.0,0.0789
dirty,10000.0,22.0,766.0,0.0789
challenge,2500.0,22.0,216.0,—


,application_id,application_date,income_sar,bureau_score,default_within_90d
0,TR-001121,2022-01-01,8938.11,584.0,0
1,TR-002836,2022-01-01,16995.65,652.0,0
2,TR-005579,2022-01-01,8353.27,734.0,0
3,TR-006881,2022-01-01,8099.85,695.0,0


Data checks passed | اجتازت البيانات الفحوص


### **3. Four essentials you will use every day**

Classification: Predicting class 0 or 1 is different from estimating a probability.

predict_proba: Returns an estimated probability between 0 and 1; it is not a guarantee.

Train / validation / test: Training is used to learn the model, validation is used to make decisions, and final evaluation is kept separate.

Confusion matrix: Shows TP, TN, FP, and FN after applying a threshold to convert probabilities into predictions.

Learning example: With 3 missed defaults and 20 false alarms under 10 × FN + FP, the simulated decision cost is 50 units. This is an educational metric, not real monetary loss or a grade deduction


### **4. Test tool compatibility**

The shared cell uses a small independent sample of 300 synthetic rows to verify that Logistic Regression, XGBoost, LightGBM, SHAP, calibration, and Optuna work correctly in the project environment.

This is a software compatibility check only. It does not select the final model and is not an assessed modeling result.

Expected result: All six checks should report PASS.


In [3]:
from readiness_checks import check_runtime

runtime_report = check_runtime(ROOT)
assert all(value == "PASS" for value in runtime_report["checks"].values())
print("Environment ready | البيئة جاهزة")

logistic: PASS
xgboost: PASS
lightgbm: PASS
shap_margin_additivity: PASS
sigmoid_calibration: PASS
optuna: PASS
Environment ready | البيئة جاهزة


### **5. Check your understanding — six ungraded questions**

1. **predict_proba returns 0.20:** B — An estimated 20% probability.
2. **Where should missing-value imputation be fitted?** C — Training rows inside each fold.
3. **Which field suggests temporal leakage?** A — Collections contact after application.
4. **A customer has repeated applications:** A — Separate customers across validation boundaries.
5. **With 3 FN, 20 FP, and weights 10 and 1, what is the simulated cost?** B — 50.
6. **Where should progress be saved before Colab closes?** C — Notebook and outputs in the repository or on the device.


In [4]:

answers = [None, None, None, None, None, None]  # Replace None with "A", "B", or "C".
answer_key = ["B", "C", "A", "A", "B", "C"]
feedback = [
    "Probability is an estimate, not a guarantee | الاحتمال تقدير لا ضمان",
    "Fit preprocessing on training rows only | تعلّم التعويض من التدريب فقط",
    "Post-application collections are unavailable at decision time | التحصيل لاحق للقرار",
    "Separate repeated customers across validation boundaries | افصل العملاء المتكررين",
    "10 × 3 + 20 = 50 simulated units | وحدات افتراضية",
    "Save the notebook and actual outputs | احفظ الدفتر والمخرجات",
]
if len(answers) != 6 or any(a not in [None, "A", "B", "C"] for a in answers):
    raise ValueError("Use six answers: A, B, C, or None.")
answered = sum(a is not None for a in answers)
correct = sum(a == expected for a, expected in zip(answers, answer_key))
for number, (answer, expected, hint) in enumerate(zip(answers, answer_key, feedback), 1):
    status = "Not answered" if answer is None else "Correct" if answer == expected else "Review"
    print(f"{number}. {status}" + (f" — {hint}" if answer is not None else ""))
print(f"Self-check: {answered}/6 answered; {correct} correct. Not graded | غير محسوب في الدرجة")

1. Not answered
2. Not answered
3. Not answered
4. Not answered
5. Not answered
6. Not answered
Self-check: 0/6 answered; 0 correct. Not graded | غير محسوب في الدرجة


### **6. Export readiness evidence**


In [5]:
import platform, zipfile

report = {"status": "READY", "dataset_version": environment["dataset_version"],
          "course_revision": COURSE_REVISION, "python": platform.python_version(),
          "elapsed_seconds": round(time.perf_counter() - RUN_STARTED, 2),
          "data_checks": "PASS", "compatibility_checks": runtime_report["checks"],
          "self_check_answered": answered, "self_check_correct": correct,
          "self_check_is_graded": False}
if IN_COLAB:
    import resource
    report["peak_process_memory_mib"] = round(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024, 1)
(ROOT / "artifacts/readiness_report.json").write_text(json.dumps(report, indent=2), encoding="utf-8")
with zipfile.ZipFile(ROOT / "artifacts/readiness_artifacts.zip", "w", zipfile.ZIP_DEFLATED) as bundle:
    for filename in ["environment.json", "data_check.json", "runtime_checks.json", "readiness_report.json"]:
        bundle.write(ROOT / "artifacts" / filename, filename)
print("READY — احفظ ملفات artifacts ونسخة الدفتر")
print(f"Elapsed: {report['elapsed_seconds']} seconds | CPU | {len(train):,} training rows")

READY — احفظ ملفات artifacts ونسخة الدفتر
Elapsed: 41.56 seconds | CPU | 10,000 training rows
